# Sultai mana-base explorer

Scores measure weighted casting opportunities only. Set source bounds and aggregate FAMILY_BOUNDS in the input cell, rebuild the table, then rank the feasible bases. COST_WEIGHTS can change casting-cost importance without rebuilding.

The model uses independent source availability and seen-type Verge enabling. Cavern production is unrestricted; sequencing and life totals are not simulated. Format filtering uses the repository's dated Standard snapshot.

The report ranks the full pool once and reuses that score snapshot for casting-cost weight sensitivity.

For the supplied deck with zero to two Opulent Palaces, use sultai_local_search.ipynb or python -m examples.sultai_optimize. This exhaustive explorer keeps a separate flexible pool without tri-lands; enabling them increases grid memory substantially.

In [ ]:
from pathlib import Path
import sys
from time import perf_counter
from math import prod
from html import escape

# Works when launched from the repository or one of its subdirectories.
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'mtg_tools.mana_opportunities.py').exists())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from IPython.display import HTML, display
from tqdm.auto import tqdm
from mtg_tools.mana_lands import ManaSource
from mtg_tools.mana_opportunities import CastingMode, Spell, SpellCost, spell_components
from mtg_tools.mana_search import ManaSpace, build_component_table


def show_table(headers, rows):
    header = ''.join(f'<th>{escape(str(h))}</th>' for h in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{escape(str(v))}</td>' for v in row) + '</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>'))


def source_name(source):
    return f'{source.kind}:{source.colors}'


def source_list(entries):
    return ', '.join(f'{n} {source_name(source)}' for source, n in entries) or 'none'


def number(value):
    return 'unbounded' if value is None else f'{float(value):.8g}'


def sensitivity_row(result):
    interval = result.optimal_interval
    below, above = result.lower, result.upper
    return (
        result.parameter, number(result.current_value), number(interval.lower),
        source_list(below.swap.incoming) if below else 'none',
        source_list(below.swap.out) if below else 'none',
        number(interval.upper),
        source_list(above.swap.incoming) if above else 'none',
        source_list(above.swap.out) if above else 'none',
    )

## Spell profiles and search space

LAND_FAMILIES supplies per-source bounds. FAMILY_BOUNDS supplies aggregate minimum/maximum counts across a land family. FIXED_LANDS supplies exact individual-source counts within the land total. All three kinds of constraint apply consistently.

Typed shocks enable conditional Verge colors when qualifying types appear among lands seen. Fixing a source count does not guarantee drawing it. Full spell profiles and their mode shares are preserved.

In [ ]:
from examples.sultai import SPELLS, MANA_MODEL

LAND_COUNT = 26
COLORS = 'UBG'
FORMAT = 'standard'
LAND_FAMILIES = {
    'basic': (0, 1), 'shock': (0, 4), 'fast': (0, 4),
    'verge': (0, 4), 'town': (0, 4),
}
FAMILY_BOUNDS = {'basic': (1, 3), 'shock': (0, 12), 'town': (0, 4)}
FIXED_LANDS = {
    ManaSource('WUBRG', 'cavern'): 4,
    ManaSource('B', 'basic'): 1,
}

## Build search tables

Set `TURNS` below to a positive integer (default 7). Opportunities are averaged from turn 1 through `TURNS`, over both play and draw positions. Turns are averaged during construction, so changing this setting requires rebuilding the table and rerunning the analysis.

Run this cell again only if you change the spell profile, bounds, or evaluation settings. The explicit search pool includes basics, typed shocks, Town/Cavern rainbow sources, and conditional Verges. Cavern restrictions are excluded.

Batch creation shares probability calculations across every base. Component bars report the expensive transforms; base bars report row assembly. The scalar backend remains available with `backend="scalar"` for exhaustive reference checks.


In [ ]:
TURNS = 7
space = ManaSpace.from_colors(
    LAND_COUNT, COLORS, families=LAND_FAMILIES, format=FORMAT,
    fixed=FIXED_LANDS, family_bounds=FAMILY_BOUNDS,
)
show_table(['Fixed source', 'Count'], [(source_name(s), n) for s, n in FIXED_LANDS.items()])
show_table(['Family', 'Minimum', 'Maximum'], [(kind, lo, hi) for kind, (lo, hi) in space.family_bounds.items()])
print(f'Exhaustive candidates: {space.candidate_count():,}')
print(f'Capacity grid: {prod(hi + 1 for _, hi in space.bounds.values()):,} cells')
started = perf_counter()
table = build_component_table(
    space, SPELLS, model=MANA_MODEL, turns=TURNS,
    progress=lambda candidates: tqdm(candidates, total=space.candidate_count(), desc='Base rows', unit='base'),
    component_progress=lambda costs: tqdm(costs, total=len(spell_components(SPELLS)[1]), desc='Components', unit='cost'),
)
print(f'{len(table.rows)} bases built in {perf_counter() - started:.2f}s')

## Casting-cost weights

Optional absolute cost-weight overrides. Changing these weights reuses the component table. Changing composition limits requires rebuilding the space and table.

In [ ]:
COST_WEIGHTS = {}  # Example: {'1B': 2}

## Analysis

The score is the normalized casting-opportunity score. Highest-scoring bases appear first. Sensitivity varies each casting-cost weight while holding the others fixed. Bounds and successor swaps are exact within the evaluated pool.

In [ ]:
started = perf_counter()
scored = table.score(COST_WEIGHTS)
ranked = scored.rank(limit=10)
optimum = ranked[0]
winner = dict(zip(space.sources, optimum.counts))

display(HTML('<h3>Optimum solution</h3>'))
show_table(['Score'], [(f'{float(optimum.score):.9f}',)])
show_table(['Source', 'Count'], [(source_name(s), n) for s, n in winner.items() if n] + [('Total', sum(winner.values()))])
display(HTML('<h3>Top 10 candidates</h3>'))
show_table(['Rank', 'Score'], [(i, f'{float(r.score):.9f}') for i, r in enumerate(ranked, 1)])

display(HTML('<h3>Casting-cost weight sensitivity</h3>'))
cost_sensitivities = {cost: scored.crossings(cost) for cost in table.costs}
sensitivity_headers = ['Parameter', 'Current', 'Lower boundary', 'IN below', 'OUT below',
                       'Upper boundary', 'IN above', 'OUT above']
show_table(sensitivity_headers, [sensitivity_row(r) for r in cost_sensitivities.values()])
analysis = dict(scored=scored, optimum=optimum, ranked=ranked, cost_sensitivities=cost_sensitivities)
print(f'Report computed in {perf_counter() - started:.2f}s')